# Modelling

The objective of this stage is to establish forecasting baselines and compare regression models for predicting energy consumption 15 minutes ahead. Models are evaluated using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE), with lower values indicating better predictive performance.

In [10]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

## Load Processed Data

In [11]:
train = pd.read_csv('../data/train.csv', parse_dates=['date'])
val = pd.read_csv('../data/validation.csv', parse_dates=['date'])
test = pd.read_csv('../data/test.csv', parse_dates=['date'])

## Naive Baseline

A naive lag-1 forecast uses the previous 15-minute energy consumption as the prediction for the current interval. This provides a simple benchmark against which more complex models can be evaluated.

In [12]:
# assumption baseline
baseline_mae = mean_absolute_error(val['Usage_kWh'], val['lag_1'])
baseline_rmse = np.sqrt(mean_squared_error(val['Usage_kWh'], val['lag_1']))
print(f'Baseline MAE: {baseline_mae}')
print(f'Baseline RMSE: {baseline_rmse}')                      

Baseline MAE: 5.7516411251212425
Baseline RMSE: 12.78130286585032


In [13]:
# evaluate persistence baseline on the test set
test_baseline_mae = mean_absolute_error(test['Usage_kWh'], test['lag_1'])
test_baseline_rmse = np.sqrt(mean_squared_error(test['Usage_kWh'], test['lag_1']))

print(f'Test Baseline MAE: {test_baseline_mae}')
print(f'Test Baseline RMSE: {test_baseline_rmse}')

Test Baseline MAE: 5.038832428238945
Test Baseline RMSE: 11.675178747990005


## Feature Selection and Preprocessing

The forecasting features consist of time-based variables, categorical operating indicators, and historical energy consumption through lag features. The date column is retained in the feature dataframe for chronological reference and plotting but is excluded from the model preprocessing pipeline.

In [14]:
# separating predictors from target
x_train = train[['date','hour','minute','month','Day_of_week','WeekStatus','lag_1','lag_2','lag_4',
                 'lag_96','lag_672']]
y_train = train['Usage_kWh']

x_val = val[['date','hour','minute','month','Day_of_week','WeekStatus','lag_1','lag_2','lag_4',
                 'lag_96','lag_672']]
y_val = val['Usage_kWh']

x_test = test[['date','hour','minute','month','Day_of_week','WeekStatus','lag_1','lag_2','lag_4',
                 'lag_96','lag_672']]
y_test = test['Usage_kWh']


## Linear Regression

Linear Regression was used as a simple machine-learning benchmark to assess whether the selected features can explain future energy consumption through a linear relationship.

In [15]:
# one-hot encoding categorical features
categorical_features = ['Day_of_week','WeekStatus']
numerical_features = ['hour','minute','month','lag_1','lag_2','lag_4','lag_96','lag_672']

# creating a preprocessor
preprocessor = ColumnTransformer(transformers = [('categorical', OneHotEncoder(handle_unknown='ignore'),
                                                  categorical_features),
                                                 ('numerical', 'passthrough', numerical_features)])

# model pipeline with linear regression
linear_pipeline = Pipeline(steps = [('preprocessor', preprocessor),
                                 ('linearmodel', LinearRegression())])

# training model
linear_pipeline.fit(x_train, y_train)
# predict on validation set
y_pred = linear_pipeline.predict(x_val)

# evaluate
linear_mae = mean_absolute_error(y_val, y_pred)
linear_rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print(f'Linear Regression MAE: {linear_mae}')
print(f'Linear Regression RMSE: {linear_rmse}')

Linear Regression MAE: 6.584016728578933
Linear Regression RMSE: 11.931708119177065


## Random Forest

Random Forest was selected to capture nonlinear relationships and interactions between the forecasting features that may not be represented by the linear model.

In [16]:
rf_pipeline = Pipeline(steps = [('preprocessor', preprocessor),
                                 ('rfmodel', RandomForestRegressor(random_state=42))])
rf_pipeline.fit(x_train, y_train)
y_pred = rf_pipeline.predict(x_val)

# evaluate
rf_mae = mean_absolute_error(y_val, y_pred)
rf_rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print(f'Random Forest MAE: {rf_mae}')
print(f'Random Forest RMSE: {rf_rmse}')

Random Forest MAE: 4.801354471387002
Random Forest RMSE: 10.067809507471294


## Model Comparison

The initial models are compared using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE). Lower values indicate better predictive performance.

In [17]:
comparison = pd.DataFrame({
    'Model': ['Naive Baseline', 'Linear Regression', 'Random Forest'],
    'MAE': [baseline_mae, linear_mae, rf_mae],
    'RMSE': [baseline_rmse, linear_rmse, rf_rmse]})
comparison

,Model,MAE,RMSE
0,Naive Baseline,5.751641,12.781303
1,Linear Regression,6.584017,11.931708
2,Random Forest,4.801354,10.067810


## Random Forest Hyperparameter Tuning

TimeSeriesSplit was used for cross-validation to preserve the temporal ordering of the observations. Hyperparameter tuning was performed using Mean Absolute Error as the evaluation metric, with the configuration producing the lowest cross-validation MAE selected as the best model.

In [18]:
tscv = TimeSeriesSplit(n_splits=3)

param_grid = {
    'rfmodel__n_estimators': [100, 200],
    'rfmodel__max_depth': [20, 30],
    'rfmodel__min_samples_leaf': [2, 5]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(x_train, y_train)

print('Best Parameters:')
print(grid_search.best_params_)

print('\nBest CV MAE:')
print(-grid_search.best_score_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best Parameters:
{'rfmodel__max_depth': 20, 'rfmodel__min_samples_leaf': 5, 'rfmodel__n_estimators': 200}

Best CV MAE:
5.24087885212572


## Tuned Model Evaluation

In [19]:
best_rf = grid_search.best_estimator_

y_val_pred = best_rf.predict(x_val)

val_mae = mean_absolute_error(y_val, y_val_pred)
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

print(f'Validation MAE: {val_mae}')
print(f'Validation RMSE: {val_rmse}')

Validation MAE: 4.561224736891662
Validation RMSE: 9.791013898949563


## Overfitting Check

Training performance is compared with validation performance to assess whether the tuned model shows evidence of overfitting.

In [20]:
# checking model overfitting
y_train_pred = best_rf.predict(x_train)

train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))

print(f'Training MAE: {train_mae}')
print(f'Training RMSE: {train_rmse}')

Training MAE: 3.420487673577543
Training RMSE: 7.257070881788715


## Test Set Evaluation

The test set provides the final evaluation of the selected model on observations that were not used during model training or hyperparameter selection.

In [21]:
y_test_pred = best_rf.predict(x_test)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f'Test MAE: {test_mae}')
print(f'Test RMSE: {test_rmse}')

Test MAE: 4.422055876223163
Test RMSE: 9.394049900677075


In [22]:
mae_improvement = (test_baseline_mae - test_mae) / test_baseline_mae * 100
rmse_improvement = (test_baseline_rmse - test_rmse) / test_baseline_rmse * 100

print(f'MAE improvement over baseline: {mae_improvement:.1f}%')
print(f'RMSE improvement over baseline: {rmse_improvement:.1f}%')

MAE improvement over baseline: 12.2%
RMSE improvement over baseline: 19.5%


In [23]:
final_comparison = pd.DataFrame({
    'Model': ['Naive Baseline', 'Tuned Random Forest'],
    'MAE': [test_baseline_mae, test_mae],
    'RMSE': [test_baseline_rmse, test_rmse]})
final_comparison

,Model,MAE,RMSE
0,Naive Baseline,5.038832,11.675179
1,Tuned Random Forest,4.422056,9.394050


## Save Final Model

The selected tuned Random Forest pipeline is saved for later use in model interpretation and application development.

In [24]:
joblib.dump(best_rf, '../models/final_random_forest.pkl')
print("Final model saved successfully.")

Final model saved successfully.


Summary: Several forecasting models were developed to predict 15-minute-ahead energy usage using historical consumption and time-based features. A naïve persistence baseline was established using the previous 15-minute consumption, followed by Linear Regression and Random Forest models. The Random Forest was subsequently tuned using time-series cross-validation on the training data while preserving chronological order. The selected model used 200 trees, a maximum depth of 20, and a minimum of 5 samples per leaf. Model performance was evaluated on validation data during development and on the previously untouched test set for final assessment. On the test set, the tuned Random Forest achieved a MAE of 4.42 kWh and RMSE of 9.39 kWh, compared with 5.04 kWh and 11.68 kWh for the persistence baseline. This represents a 12.2% improvement in MAE and a 19.5% improvement in RMSE. The final model was saved for subsequent interpretation and application development.